# PySpark CDC — Kafka JSON Pattern

CDC JSON parsing pattern. The first part is fully runnable with static JSON data. The Kafka readStream block is included as the direct streaming equivalent.

## 00 — Spark setup and sample data

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("cdc-kafka-bronze-silver")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/cdc-kafka-bronze-silver_warehouse")
    .getOrCreate()
)

spark.version

'4.0.2'

In [2]:
json_messages = spark.createDataFrame([
    ('{"customer_id":1,"name":"Alice","email":"alice@new.com","op":"U","event_time":"2026-05-02T10:05:00"}',),
    ('{"customer_id":2,"name":null,"email":null,"op":"D","event_time":"2026-05-02T10:10:00"}',),
    ('{"customer_id":3,"name":"Carol","email":"carol@mail.com","op":"I","event_time":"2026-05-02T10:15:00"}',),
], ["value"])

json_messages.show(truncate=False)

[Stage 0:>                                                          (0 + 1) / 1]

+-----------------------------------------------------------------------------------------------------+
|value                                                                                                |
+-----------------------------------------------------------------------------------------------------+
|{"customer_id":1,"name":"Alice","email":"alice@new.com","op":"U","event_time":"2026-05-02T10:05:00"} |
|{"customer_id":2,"name":null,"email":null,"op":"D","event_time":"2026-05-02T10:10:00"}               |
|{"customer_id":3,"name":"Carol","email":"carol@mail.com","op":"I","event_time":"2026-05-02T10:15:00"}|
+-----------------------------------------------------------------------------------------------------+



## 01 — Parse JSON CDC events

In [3]:
cdc_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("op", StringType(), False),
    StructField("event_time", StringType(), False),
])

parsed_cdc = (
    json_messages
    .select(F.from_json("value", cdc_schema).alias("cdc"))
    .select("cdc.*")
    .withColumn("event_time", F.to_timestamp("event_time"))
)

parsed_cdc.orderBy("event_time").show(truncate=False)

+-----------+-----+--------------+---+-------------------+
|customer_id|name |email         |op |event_time         |
+-----------+-----+--------------+---+-------------------+
|1          |Alice|alice@new.com |U  |2026-05-02 10:05:00|
|2          |NULL |NULL          |D  |2026-05-02 10:10:00|
|3          |Carol|carol@mail.com|I  |2026-05-02 10:15:00|
+-----------+-----+--------------+---+-------------------+



## 02 — Kafka readStream equivalent

In [4]:
# kafka_bootstrap_servers = "kafka:9092"
# topic = "customers_cdc"
#
# kafka_raw = (
#     spark.readStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
#     .option("subscribe", topic)
#     .option("startingOffsets", "earliest")
#     .load()
# )
#
# parsed_stream = (
#     kafka_raw
#     .select(F.col("value").cast("string").alias("value"))
#     .select(F.from_json("value", cdc_schema).alias("cdc"))
#     .select("cdc.*")
#     .withColumn("event_time", F.to_timestamp("event_time"))
# )

## 03 — Apply parsed CDC events

In [5]:
customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("updated_at", StringType(), False),
])

customers = spark.createDataFrame([
    (1, "Alice", "alice@old.com", "2026-05-02T09:00:00"),
    (2, "Bob", "bob@mail.com", "2026-05-02T09:00:00"),
], customers_schema).withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)

from pyspark.sql.window import Window

latest_window = Window.partitionBy("customer_id").orderBy(F.col("event_time").desc())

latest_cdc = (
    parsed_cdc
    .withColumn("rn", F.row_number().over(latest_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

delete_keys = latest_cdc.filter(F.col("op") == "D").select("customer_id")
after_deletes = customers.join(delete_keys, on="customer_id", how="left_anti")

upserts = (
    latest_cdc
    .filter(F.col("op").isin("I", "U"))
    .select(
        "customer_id",
        "name",
        "email",
        F.col("event_time").alias("updated_at")
    )
)

customers_after_cdc = (
    after_deletes
    .join(upserts.select("customer_id"), on="customer_id", how="left_anti")
    .unionByName(upserts)
)

customers_after_cdc.orderBy("customer_id").show(truncate=False)

+-----------+-----+--------------+-------------------+
|customer_id|name |email         |updated_at         |
+-----------+-----+--------------+-------------------+
|1          |Alice|alice@new.com |2026-05-02 10:05:00|
|3          |Carol|carol@mail.com|2026-05-02 10:15:00|
+-----------+-----+--------------+-------------------+

